In [ ]:
import re
import imaplib
import email
from email.header import decode_header
import getpass # To securely ask for the password

def fetch_emails_from_gmail(username, password):

    # Gmail IMAP server
    imap_server = "imap.gmail.com"
    target_senders = ["vitlions2026@vitbhopal.ac.in", "placementoffice@vitbhopal.ac.in"]

    try:
        # Connect to the server
        print("Connecting to Gmail...")
        mail = imaplib.IMAP4_SSL(imap_server)

        # Login
        print("Logging in...")
        mail.login(username, password)

        # Select the inbox
        mail.select("inbox")

        # --- OPTIMIZATION: Search on the server before downloading ---
        print("Searching for relevant emails on the server (this is much faster)...")
        all_email_ids = set()
        for sender in target_senders:
            print(f"-> Searching for emails from: {sender}")
            status, messages = mail.search(None, f'(FROM "{sender}")')
            if status == 'OK':
                email_ids = messages[0].split()
                if email_ids:
                    print(f"   Found {len(email_ids)} emails.")
                    all_email_ids.update(email_ids)

        if not all_email_ids:
            print("\nNo emails found from any of the specified senders.")
            mail.logout()
            return []

        unique_email_ids = list(all_email_ids)
        print(f"\nFound a total of {len(unique_email_ids)} relevant emails to download and process.")

        emails_data = []

        # Iterate through the MUCH SMALLER list of email IDs
        for email_id in unique_email_ids:
            status, msg_data = mail.fetch(email_id, "(RFC822)")
            if status != 'OK':
                continue

            for response_part in msg_data:
                if isinstance(response_part, tuple):
                    msg = email.message_from_bytes(response_part[1])

                    from_header, encoding = decode_header(msg.get("From"))[0]
                    if isinstance(from_header, bytes):
                        from_header = from_header.decode(encoding if encoding else "utf-8")

                    subject, encoding = decode_header(msg.get("Subject"))[0]
                    if isinstance(subject, bytes):
                        subject = subject.decode(encoding if encoding else "utf-8")

                    body = ""
                    if msg.is_multipart():
                        for part in msg.walk():
                            if part.get_content_type() == "text/plain":
                                try:
                                    body = part.get_payload(decode=True).decode()
                                    break
                                except: pass
                    else:
                        try:
                            body = msg.get_payload(decode=True).decode()
                        except: pass

                    full_content = f"Subject: {subject}\n\n{body}"
                    emails_data.append({'from': from_header, 'content': full_content})

        mail.logout()
        print("Logout successful.")
        return emails_data

    except imaplib.IMAP4.error as e:
        print(f"IMAP Error: {e}")
        print("Login failed. Please check your email, App Password, and ensure IMAP is enabled in your Gmail settings.")
        return []

import re

def parse_registration_numbers(emails_data):

    registration_number_pattern = r'\b\d{2}[A-Za-z]{3}1\d{4}\b'

    # --- FILTERS ---
    target_senders = ["vitlions2026@vitbhopal.ac.in", "placementoffice@vitbhopal.ac.in"]

    # Regex that ensures BOTH "Selection List" and "2026 Batch" appear in text
    selection_pattern = re.compile(r"selection\s*list.*2026\s*batch", re.IGNORECASE | re.DOTALL)

    extracted_numbers = []

    print("\n--- Starting Email Analysis ---")
    for i, email_data in enumerate(emails_data):
        print(f"\nProcessing Email #{i+1}...")

        # Ensure sender is in target list
        if not any(sender in email_data['from'].lower() for sender in target_senders):
            print("-> Sender not in target list. Skipping.")
            continue

        # Subject + body combined
        content_to_check = email_data['content']

        # Check if both "Selection List" and "2026 Batch" exist
        if selection_pattern.search(content_to_check):
            print("-> Found 'Selection List' and '2026 Batch' in subject or body.")

            # Extract registration numbers
            found_numbers = re.findall(registration_number_pattern, content_to_check)
            if found_numbers:
                print(f"--> Extracted Numbers: {found_numbers}")
                extracted_numbers.extend(found_numbers)
            else:
                print("--> No valid registration numbers found in this email.")
        else:
            print("-> Required keywords not found. Skipping.")

    print("\n--- Analysis Complete ---")
    return list(set(extracted_numbers))


# --- Main execution ---
if __name__ == "__main__":
    user_email = input("Enter your Gmail address: ")
    app_password = getpass.getpass("Enter your 16-digit Google App Password: ")

    live_emails_data = fetch_emails_from_gmail(user_email, app_password)

    if live_emails_data:
        final_registration_numbers = parse_registration_numbers(live_emails_data)

        if final_registration_numbers:
            print(f"\n Final list of unique registration numbers from selection emails:")
            for number in sorted(final_registration_numbers):
                print(f"- {number}")
        else:
            print("\n No registration numbers could be extracted based on the new criteria.")
    else:
        print("\nCould not fetch any emails to analyze.")



Streaming output truncated to the last 5000 lines.

Processing Email #3317...
-> Required keywords not found. Skipping.

Processing Email #3318...
-> Required keywords not found. Skipping.

Processing Email #3319...
-> Required keywords not found. Skipping.

Processing Email #3320...
-> Required keywords not found. Skipping.

Processing Email #3321...
-> Required keywords not found. Skipping.

Processing Email #3322...
-> Required keywords not found. Skipping.

Processing Email #3323...
-> Required keywords not found. Skipping.

Processing Email #3324...
-> Required keywords not found. Skipping.

Processing Email #3325...
-> Required keywords not found. Skipping.

Processing Email #3326...
-> Required keywords not found. Skipping.

Processing Email #3327...
-> Required keywords not found. Skipping.

Processing Email #3328...
-> Found 'Selection List' and '2026 Batch' in subject or body.
--> No valid registration numbers found in this email.

Processing Email #3329...
-> Required keywor

In [ ]:
print(len(final_registration_numbers))

84
